# 03 — BART (abstractive)

**BART** (Lewis et al., 2019) è un transformer encoder-decoder pre-addestrato come denoising
autoencoder; il checkpoint usato qui, **`facebook/bart-large-cnn`**, è la versione fine-tuned per
la summarization sul dataset CNN/DailyMail (notizie single-document). A differenza di TextRank e
LexRank il riassunto è *generato* parola per parola, non estratto dal testo.

Questo notebook opera **solo sul campione condiviso** (`SCOPE` non esiste: la generazione
neurale sull'intero dataset non è praticabile). Su CPU contare ~30–90 s a esempio; con GPU NVIDIA
(rilevata automaticamente) pochi secondi a esempio.

**Limite noto**: l'input viene troncato ai primi **1024 token** (il massimo del modello). I
documenti Multi-News superano spesso questa soglia (mediana ~1.300 parole), quindi il modello
«vede» solo l'inizio del cluster di articoli — è la prassi standard per questi checkpoint e vale
allo stesso modo per PEGASUS (notebook 04), quindi il confronto resta equo.

Riassunti salvati incrementalmente in `results/summaries/bart_sample.tsv` con **ripresa** dopo
interruzione; metriche ricalcolabili dai soli file salvati.

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer sentencepiece

In [ ]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO     = 'bart'
SCOPE      = 'sample'    # questo notebook gira solo sul campione
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti
CHECKPOINT = 'facebook/bart-large-cnn'
MAX_LEN    = 300         # lunghezza massima (token) del riassunto generato
NUM_BEAMS  = 4
MAX_INPUT  = 1024        # limite di input del modello

BASE   = su.trova_base_dir()
P      = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'
DEVICE = su.rileva_device()

print(f'Checkpoint : {CHECKPOINT}')
print(f'Device     : {DEVICE}')
print(f'Output     : {OUT_PATH}')

## Generazione dei riassunti

La funzione `summ_ext_bart` di pyAutoSummarizer ricarica il modello **a ogni chiamata** e non usa
mai la GPU; qui replichiamo esattamente la sua logica (stessi parametri di generazione: beam
search a 4 beam, `early_stopping`, troncamento dell'input a 1024 token) ma caricando il modello
**una sola volta** e spostandolo sul `DEVICE` rilevato. Il separatore `` ||||| `` tra articoli
viene sostituito con un newline prima della tokenizzazione.

In [ ]:
import torch
from transformers import BartForConditionalGeneration, BartTokenizer

tokenizer = BartTokenizer.from_pretrained(CHECKPOINT)
modello   = BartForConditionalGeneration.from_pretrained(CHECKPOINT).to(DEVICE).eval()

def genera(documento):
    inputs = tokenizer([documento], max_length=MAX_INPUT, truncation=True,
                       padding='longest', return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = modello.generate(inputs['input_ids'], attention_mask=inputs['attention_mask'],
                               num_beams=NUM_BEAMS, max_length=MAX_LEN, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

esempi    = su.carica_campione(SAMPLE_PATH)
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT, etichetta='BART ')
scrittore.chiudi()

## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/bart_sample_per_example.csv` e `…_aggregate.json` (medie
complessive e per split).

In [ ]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = su.carica_campione(SAMPLE_PATH)
config = {'checkpoint': CHECKPOINT, 'max_len': MAX_LEN, 'num_beams': NUM_BEAMS,
          'max_input_tokens': MAX_INPUT, 'device': DEVICE,
          'libreria': 'transformers (parametri di pyAutoSummarizer.summ_ext_bart)'}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))

## Ispezione qualitativa

In [ ]:
su.mostra_esempi(su.carica_campione(SAMPLE_PATH), riassunti, quanti=2)